In [ ]:
%load_ext autoreload 
%autoreload 2
from oat.data_module.cosmo_peakweather_dataset import CosmoPeakWeatherConfig, CosmoPeakWeatherDataset
from oat.data_module import transform
from oat.data_module.data_split import create_splits
import matplotlib.pyplot as plt
import yaml
from omegaconf import OmegaConf
from hydra import compose, initialize
import torch

with initialize(config_path="../configs/", version_base=None):
    cfg = compose(config_name="config")
#train_dataset, val_dataset, test_dataset = create_splits(dataset,

In [ ]:
dataset = CosmoPeakWeatherDataset(cfg.dataset)

In [ ]:
import time
from tqdm import tqdm

start_time = time.time()
num_samples = 100
for i in tqdm(range(num_samples)):
    sample = dataset[i]
end_time = time.time()
print(f"Time taken for {num_samples} samples: {end_time - start_time} seconds")
print(f"Average time per sample: {(end_time - start_time) / num_samples} seconds")

In [ ]:
train_dataset, val_dataset, test_dataset = create_splits(dataset, **cfg.split)
train_loader = torch.utils.data.DataLoader(train_dataset, shuffle=True, **cfg.dataloader)
val_loader = torch.utils.data.DataLoader(val_dataset, shuffle=False, **cfg.dataloader)
test_loader = torch.utils.data.DataLoader(test_dataset, shuffle=False, **cfg.dataloader)

In [ ]:
input_transform, output_transform = train_dataset.get_transform(num_data_points=10, dataloader=train_loader)

In [ ]:
transformed_sample = input_transform(train_dataset[0])
transformed_sample

# Investigate data point

In [ ]:
data_point = input_transform(dataset[0])


for ivar, var in enumerate(config.nwp_vars + config.topo_vars):
    plt.figure(figsize=(8, 6))
    plt.title(f"{var} at final time step")
    im = plt.pcolor(dataset.ref_nwp_ds.x, dataset.ref_nwp_ds.y, 
               data_point['global_data']['global_field'][ivar], cmap='viridis')
    plt.scatter(data_point['observation_data']['coords'][:, 0], data_point['observation_data']['coords'][:, 1],
                c=data_point['observation_data']['obs'][:,-1], label='Stations')
    
    plt.colorbar(im, label=f"{var} value")
    plt.xlabel("X Coordinate")
    plt.show()

In [ ]:
# test iteration speed
import numpy as np
import time
start_time = time.time()
observation_list = []
for i in range(100):
    obs = dataset[i]['observation_data']['obs']
    mask = dataset[i]['observation_data']['mask_obs']
    obs[mask==0] = np.nan
    observation_list.append(obs[None])
    
end_time = time.time()
print(f"Time taken to iterate over 100 samples: {end_time - start_time} seconds")

In [ ]:
import torch
obs_tensor = torch.cat(observation_list, dim=0)

plt.plot(obs_tensor[:,:, 3])
plt.show()